# 3. Autovectorization

In [1]:
# heres a simple convolution function
import jax
import jax.numpy as jnp

def convolve(x, w):
    output = []
    for i in range(1, len(x) - 1):
        output.append(jnp.dot(x[i-1:i+2], w))
    return jnp.array(output)

now what if we want to apply this function to a batch of weights `w` to a batch of vectors `x`?

In [3]:
x = jnp.arange(5)
w = jnp.array([2., 3., 4.])

xs = jnp.stack([x, x])
ws = jnp.stack([w, w])

In [5]:
def manually_batched_convolve(xs, ws):
    output = []
    for i in range (xs.shape[0]):
        output.append(convolve(xs[i], ws[i]))
    return jnp.stack(output)

manually_batched_convolve(xs, ws)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

above works, but what's the problem? it's not very efficient ecause it's not vectorized - normally, we'd have to rewrite the function so that it's in vectorized form. we can do this, but as the complexity of a function increases, this can be messy and error-prone.

## 3.1 Welcome Autovec and vmap!

`jax.vmap` generates a vectorized implementation of a function automatically:

In [6]:
auto_batch_convolve = jax.vmap(convolve)
auto_batch_convolve(xs, ws)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

> 
> note: `in_axes`: by default, vmap assumes the **first axis** is the batch.
>
> for example:
>
> `xs.shape = (32, 100)`
>
> so it means that axis 0 -> 32 examples...
>
> but if the data is stored as something like `xs.shape = (100, 32)` - now the batch dimension is axis 1 
>
> so we can write `in_axes = 1, out_axes = 1`

> note 2:
>Why out_axes=1?
>
>Suppose each output has length 3.
>
>Normally,
>
>output1
>output2
>
>would be stacked as
>
>(2,3)
>
>If you instead want
>
>(3,2)
>
>you tell JAX
>
>out_axes=1
>
>which says
>
>"Put the batch dimension in axis 1."

Finally, the `None` argument in vmapis for when we're working with **one filter** over many signals.

In [9]:
# for example if we have xs(1000, 100) and w(10,)

jax.vmap(convolve, in_axes = [0, None]) # i.e. batch argument 1 over axis 0 and don't batch argument 2

# so if we have like f(a, b) then vmap(f, in_axes = [0, None]) would be like above.

<function __main__.convolve(x, w)>